# Linear Regression to predict Game Margin
## By: Ryan O'Farrell


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
import altair as alt
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

alt.renderers.enable('notebook')

Read in data via loop from 2003-2019 seasons.

In [ ]:
first_run = True
for season in range(2003,2020):
    if first_run:
        raw_data = pd.read_csv('data/predate_games/'+str(season)+'.csv')
        first_run = False
    else:
        data_to_append = pd.read_csv('data/predate_games/'+str(season)+'.csv')
        raw_data = raw_data.append(data_to_append, sort=False)
        del data_to_append

Let's make a copy of the raw data so we can always come back and rerun from here

In [ ]:
df = raw_data.copy()
df.shape

The data contains a record from each team's perspective in the game, so there are actually two times as many records as there should be. Let's get to just one record per game by:

- Keeping the home team
- Keeping the earlier alphabetical team when the location is neutral.

In [ ]:
df = df.loc[(df['Tm1_Loc'] == 'H') | ((df['Tm1_Loc'] == 'N') & (df['Tm1'] < df['Tm2']))]
df.shape

Exactly half of the records remain. There also are games early-on in the season, where the teams do not have a large sample size of prior-games to have any meaningful data. Let's see what that looks like:

In [ ]:
df[['Tm1_TmGame', 'Tm2_TmGame']].describe()

The min of 1 illustrates the point. Let's set a threshold of 10 previous games by both teams as a large enough sample size to keep the record.

In [ ]:
game_threshold = 10
df = df.loc[((df['Tm1_TmGame'] >= game_threshold) & (df['Tm2_TmGame'] >= game_threshold))]
df.shape

Now we are down to 56,113 records - still quite a few! Let's look at the field we want to predict: Tm1_GameMargin:

In [ ]:
y = df['Tm1_GameMargin']
del df['Tm1_GameMargin']
y.describe()

In [ ]:
np_bins = np.arange(-60,76,1)
hist, bin_edges = np.histogram(y, bins = np_bins)
chart_data = pd.DataFrame({
    'GameMargin': bin_edges[:-1],
    'count': hist
})


alt.Chart(
    chart_data
).mark_bar().encode(
    x = alt.X('GameMargin'),
    y='count'
)

In [ ]:
stats.probplot(y, dist='norm', plot=plt)
plt.show()

Data looks normal with a mean of approx. 3.6; remember we kept mostly home teams so this could be considered the home court advantage. Note the dip in the distribution at 0 points - since games don't end in ties, no margins should be 0

In [ ]:
assert hist[np.where(np_bins == 0)][0] == 0, 'Some games are ties!!'

The fields in the data are summations of the key metrics for each game played prior to the given game. We would rather have normalized values, by game, by 40 minutes, and by possession.

In [ ]:
METRICS = ['PF', 'Margin',
            'FGM', 'FGA',
            'FG3M', 'FG3A',
            'FG2M', 'FG2A',
            'FTA', 'FTM',
            'Ast', 'ORB',
            'DRB', 'TRB',
            'TO', 'Stl',
            'Blk', 'Foul']
for tm in ['Tm1_', 'Tm2_']:
    for perspective in ['Tm', 'Opp']:
        for metric in METRICS:
            df[tm+perspective+metric+'perPoss'] = df[tm+perspective+metric] / df[tm+perspective+'Poss']
            df[tm+perspective+metric+'per40'] = df[tm+perspective+metric] / df[tm+perspective+'Mins'] * 40
            df[tm+perspective+metric+'perGame'] = df[tm+perspective+metric] / df[tm+perspective+'Game']
            
    df[tm+'TmPossperGame'] = df[tm+'TmPoss'] / df[tm+'TmGame']
    df[tm+'TmPossper40'] = df[tm+'TmPoss'] / df[tm+'TmMins'] * 40

Let's build a function that will take any number of predictors and make a linear regression from it, and output some key metrics

In [ ]:
def run_regression(predictor_columns, data = df, _y = y):
    X = df[predictor_columns]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    regressor = LinearRegression()  
    regressor.fit(X_train, y_train) #training the algorithm 
    y_pred = regressor.predict(X_test)
    compared_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    compared_df['Error_Abs'] = abs(compared_df['Actual'] - compared_df['Predicted'])
    compared_df['Correct_Winner'] = compared_df['Actual']*compared_df['Predicted'] > 0

    print(f"With predictors: {predictor_columns}")
    print(compared_df.head(),'\n')
    print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))  
    print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))  
    print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
    print('R^2:', metrics.r2_score(y_test, y_pred))
    print(f"Correct winner rate: {sum(compared_df['Correct_Winner'])/len(compared_df)}")
    print('\n')
    
    return compared_df

A quick test run of our function

In [ ]:
out = run_regression(['Tm1_TmMarginper40'])

The function works, and spits out some nice result strings. Note that using just Tm1's margin of victory value gets our MAE to approx. 9.8 points.

The next question would be: is 9.8 points good?? Well, for this analysis we will use Vegas game lines as context to see if our models are good or bad. Let's see what that looks like:

In [ ]:
from db import get_db
DB = get_db()
vegasgames = pd.DataFrame(
    list(
        DB.games.find(
            {'GameVegasLine':{'$exists' : True}},
            {'_id': 0,'GameVegasLine': 1, 'OppMargin': 1, 'TmLoc': 1, 'TmName': 1, 'OppName': 1}
        )
    )
)

In [ ]:
vegasgames.describe()

In [ ]:
vegasgames.head()

Note that some games do not have a game line, and we have the same 'double record' problem as before, let's address those.

In [ ]:
vegasgames = vegasgames.loc[np.isnan(vegasgames['GameVegasLine']) == False]
vegasgames = vegasgames.loc[(vegasgames['TmLoc'] == 'H') | ((vegasgames['TmLoc'] == 'N') & (vegasgames['TmName'] < vegasgames['OppName']))]

In [ ]:
vegasgames.describe()

We can see that home teams are favored by ~4.0 points, which is not much different than the average home margin of victory.

Now let's get some comparable metrics if we were to use Vegas as our predictor.

In [ ]:
print('Mean Absolute Error:', metrics.mean_absolute_error(vegasgames['OppMargin'], vegasgames['GameVegasLine']))  
print('Mean Squared Error:', metrics.mean_squared_error(vegasgames['OppMargin'], vegasgames['GameVegasLine'])) 
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(vegasgames['OppMargin'], vegasgames['GameVegasLine'])))
print('R^2:', metrics.r2_score(vegasgames['OppMargin'], vegasgames['GameVegasLine']))
vegasgames['Correct_Winner'] = vegasgames['OppMargin']*vegasgames['GameVegasLine'] > 0
print(f"Correct winner rate: {sum(vegasgames['Correct_Winner'])/len(vegasgames)}")
print('\n')

Interesting - even vegas can only get a MAE of approx. 8.29 points, and only correctly predicts the winner 72.7% of the time.

With that in mind, let's try a few different predictors for our linear regression:

In [ ]:
def get_included_columns(included_predictors):
    included_columns = []

    for prefix in ['Tm1_', 'Tm2_']:
        for pred in included_predictors:
            included_columns.append(prefix+pred)
    return included_columns

In [ ]:
included_predictors = get_included_columns(['TmMarginper40'])
out = run_regression(included_predictors)

Simply using each team's margin of victory coming in results in a MAE of 8.7 and a correct winner rate of 70.8%!


Let's see which normalization (per40, perGame, or perPoss) yields the best results...

In [ ]:
included_predictors = get_included_columns(['TmMarginperPoss'])
out = run_regression(included_predictors)

In [ ]:
included_predictors = get_included_columns(['TmMarginperGame'])
out = run_regression(included_predictors)

Looks like per40 stats are the best. Let's stick with those and dig deeper into various other metrics as predictors

In [ ]:
included_predictors = get_included_columns(['TmPFper40', 'OppPFper40'])
out = run_regression(included_predictors)

In [ ]:
preds = ['TmPossper40']
for metric in METRICS:
    for prefix in ['Tm', 'Opp']:
        if metric not in ['Margin', 'PF']:
            preds.append(prefix+metric+'per40')

included_predictors = get_included_columns(preds)
out = run_regression(included_predictors)

Our last run, using every metric except Margins and PFs results in a MAE of 8.65 and a correct winner rate of 71% - not far from Vegas's 8.29 and 72.7%.


Now let's opponent-adjust all the metrics.



In [ ]:
def opponent_adjust_metrics(data = df, mets=METRICS):
    for persp in ['Tm1_', 'Tm2_']:
        for PREFIX in ['Tm', 'Opp']:
            OTHER_PREFIX = 'Opp' if PREFIX == 'Tm' else 'Tm'
            for METRIC in mets:
                for DENOM in ['per40', 'perGame', 'perPoss']:
                    NORMALIZE_CONST = 40 if DENOM == 'per40' else 1
                    DENOM_FIELD = 'Mins' if DENOM == 'per40' else DENOM[-4:]
                    data[persp+'OA_'+PREFIX+METRIC+DENOM] = \
                        (data[persp+PREFIX+METRIC+DENOM]) - \
                        (
                            (data[persp+'OppSum_'+OTHER_PREFIX+METRIC] - data[persp+PREFIX+METRIC]) /
                            (data[persp+'OppSum_'+OTHER_PREFIX+DENOM_FIELD] - data[persp+PREFIX+DENOM_FIELD])
                        ) * NORMALIZE_CONST
    return data

In [ ]:
df = opponent_adjust_metrics()

Let's try some more regressions, using opponent-adjusted metrics

In [ ]:
included_predictors = get_included_columns(['OA_TmMarginper40']) # This is essentially margin
out = run_regression(included_predictors)

Wow - already our MAE has dipped to 8.54 with a correct winner rate of 71.6%, only using each team's opponent-adjusted margin of victory!

In [ ]:
preds = []
for metric in METRICS:
    for prefix in ['Tm', 'Opp']:
        if metric not in ['Margin', 'PF']:
            preds.append('OA_'+prefix+metric+'per40')

included_predictors = get_included_columns(preds)
out = run_regression(included_predictors)

Let's compare the linear regression results to a small, simple random forest

In [ ]:
def run_rf_regression(predictor_columns, data = df, _y = y):
    X = df[predictor_columns]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    rf_regr = RandomForestRegressor(
        n_estimators = 10, 
        criterion = 'mae', 
        max_depth=3, 
        random_state=0, 
        verbose=1, 
        n_jobs=-1,
        min_samples_split = 4
    )
    rf_regr.fit(X_train, y_train) #training the algorithm 
    y_pred = rf_regr.predict(X_test)
    compared_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    compared_df['Error_Abs'] = abs(compared_df['Actual'] - compared_df['Predicted'])
    compared_df['Correct_Winner'] = compared_df['Actual']*compared_df['Predicted'] > 0

    print(f"With predictors: {predictor_columns}")
    print(compared_df.head(),'\n')
    print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))  
    print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))  
    print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
    print('R^2:', metrics.r2_score(y_test, y_pred))
    print(f"Correct winner rate: {sum(compared_df['Correct_Winner'])/len(compared_df)}")
    print('\n')
    
    return compared_df


In [ ]:
included_predictors = get_included_columns(['OA_TmMarginper40']) 
out = run_rf_regression(included_predictors)

Interestingly, the random forest did worse than a simple regression on the same predictors...

I know that running the random forest with more predictors will take loads of time, so that's a task for another day (and likely will require me to shift the workload to the cloud)

Next steps will be to see if changing the game threshold has an impact on predictability, and tie that to an analysis of factors where my predictions are more accurate (late-season games, certain margins I am very confident about, etc.). For example, if my model is better than vegas in any subset, that could be a place where I focus my future explorations.